In [12]:
import numpy as np
import pandas as pd
import math

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier,HistGradientBoostingClassifier


In [13]:
df=pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')


In [14]:
# Basic Dataset Overview
print(f'Data rows and columns: {df.shape}')
print(f'{df.info()}')
print(f'Checking missing value:\n{df.isna().sum()}')
for col in df.columns:
  if df[col].dtype == 'object':
    print(f'{col}: {df[col].nunique()} unique {col} - {df[col].unique()}  without missing values. No cleaning is needed.')
  elif df[col].dtype == 'int64' or df[col].dtype == 'float64':
    print(f'{col}: numeric values without missing values. No cleaning is needed.')

# Validate any negative values in numeric variables
df.describe()

Data rows and columns: (700000, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
count,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000
mean,349999.500000,50.359734,2.072411,80.230803,5.963695,7.002200,6.012733,25.874684,0.858766,116.294193,75.440924,70.167749,186.818801,53.823214,102.905854,123.081850,0.149401,0.181990,0.030324,0.623296
std,202072.738554,11.655520,1.048189,51.195071,1.463336,0.901907,2.022707,2.860705,0.037980,11.010390,6.825775,6.938722,16.730832,8.266545,19.022416,24.739397,0.356484,0.385837,0.171478,0.484560
min,0.000000,19.000000,1.000000,1.000000,0.100000,3.100000,0.600000,15.100000,0.680000,91.000000,51.000000,42.000000,117.000000,21.000000,51.000000,31.000000,0.000000,0.000000,0.000000,0.000000
25%,174999.750000,42.000000,1.000000,49.000000,5.000000,6.400000,4.600000,23.900000,0.830000,108.000000,71.000000,65.000000,175.000000,48.000000,89.000000,106.000000,0.000000,0.000000,0.000000,0.000000
50%,349999.500000,50.000000,2.000000,71.000000,6.000000,7.000000,6.000000,25.900000,0.860000,116.000000,75.000000,70.000000,187.000000,54.000000,103.000000,123.000000,0.000000,0.000000,0.000000,1.000000
75%,524999.250000,58.000000,3.000000,96.000000,7.000000,7.600000,7.400000,27.800000,0.880000,124.000000,80.000000,75.000000,199.000000,59.000000,116.000000,139.000000,0.000000,0.000000,0.000000,1.000000
max,699999.000000,89.000000,9.000000,747.000000,9.900000,9.900000,16.500000,38.400000,1.050000,163.000000,104.000000,101.000000,289.000000,90.000000,205.000000,290.000000,1.000000,1.000000,1.000000,1.000000


In [15]:
# Identify target type
target= 'diagnosed_diabetes'
id = 'id'

if df[target].dtype == 'object' or df[target].nunique() <= 10:
  target_type = 'classification'
else:
  target_type='regression'

print(f'Machine Learning Problem {target_type}')

# Identify feature type
cat_features = df.select_dtypes(include='object', exclude=['int64','float64']).columns.tolist()
num_features = df.select_dtypes(include=['int64','float64'],exclude='object').columns.tolist()
num_features = list(filter(lambda x: (x!= 'id') and (x!='diagnosed_diabetes'),num_features)) # Filter id and target column

# Separate features and target
X = df.drop(target,axis=1)
y = df[target]


Machine Learning Problem classification


In [16]:
from sklearn.metrics import roc_auc_score
def Pipeline_Model(model,X_train,y_train,X_test,X_test_ids):
  # pipeline
  preprocessor = ColumnTransformer(
      transformers= [
          ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),cat_features),
          ('num', StandardScaler(),num_features)
      ]
  )

  pipeline = Pipeline(
      steps=(
          ('preprocessor',preprocessor),
          ('model',model)
      )
  )

  pipeline.fit(X_train,y_train)

  # cross-validation
  y_train_proba  = cross_val_predict(
      pipeline,
      X_train,
      y_train,
      cv=5,
      method='predict_proba'
  )[:,1]

  y_test_proba = pipeline.predict_proba(X_test)[:, 1]
  y_test_pred = pipeline.predict(X_test)

  df_submission = pd.DataFrame({'id': X_test_ids , target:np.round(y_test_proba,2)})
  df_submission.to_csv(model.__class__.__name__ + '.csv',index=False)
  print(df_submission)

In [17]:
X_test_ids = df_test['id']
X_test = df_test.drop('id',axis=1)

hgb_model = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50
)
Pipeline_Model(hgb_model, X, y, X_test, X_test_ids)

            id  diagnosed_diabetes
0       700000                0.50
1       700001                0.68
2       700002                0.79
3       700003                0.40
4       700004                0.93
...        ...                 ...
299995  999995                0.74
299996  999996                0.68
299997  999997                0.55
299998  999998                0.60
299999  999999                0.62

[300000 rows x 2 columns]
